# Preparación de Datasets 
Para empezar la práctica, aquí cargamos lo básico. Usamos el dataset Iris de sklearn como nuestro benchmark porque ya viene limpio y nos sirve para comprobar que la lógica del código funciona bien. También generamos un dataset sintético al que le agregamos ruido a propósito para ver cómo se comportan los modelos en un escenario que no es perfecto. Además, dejamos listas las funciones para partir los datos usando Hold-Out (70/30) y K-Fold (con k=5).

In [7]:
import numpy as np
from sklearn.datasets import load_iris, make_classification
from sklearn.metrics import accuracy_score

def particion_hold_out(X, y):
    indices = np.random.permutation(len(X))
    corte = int(len(X) * 0.7)
    train_idx = indices[:corte]
    test_idx = indices[corte:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def evaluacion_kfold(X, y, k, funcion_modelo):
    indices = np.random.permutation(len(X))
    X_revuelto = X[indices]
    y_revuelto = y[indices]
    
    tamano_fold = len(X) // k
    precisiones = []
    
    for i in range(k):
        inicio = i * tamano_fold
        if i < k - 1:
            fin = inicio + tamano_fold
        else:
            fin = len(X)
            
        X_test = X_revuelto[inicio:fin]
        y_test = y_revuelto[inicio:fin]
        
        X_train = np.concatenate((X_revuelto[:inicio], X_revuelto[fin:]))
        y_train = np.concatenate((y_revuelto[:inicio], y_revuelto[fin:]))
        
        y_pred = funcion_modelo(X_train, y_train, X_test)
        precision = accuracy_score(y_test, y_pred)
        precisiones.append(precision)
        
    return np.mean(precisiones)

dataset_benchmark = load_iris()
X_bench = dataset_benchmark.data
y_bench = dataset_benchmark.target

X_sintetico, y_sintetico = make_classification(
    n_samples=300, 
    n_features=4, 
    n_classes=3, 
    n_informative=4, 
    n_redundant=0, 
    flip_y=0.15, 
    random_state=42
)

print("Dataset: CARGADO; Funciones: CARGADO")

Dataset: CARGADO; Funciones: CARGADO


# Modelo de Distancia Mínima
En esta celda se realizo el modelo desde cero sin usar clasificadores preconstruidos. La lógica que se uso se divide en dos pasos: primero, en la parte de entrenamiento, el algoritmo agrupa los datos según su clase y calcula el promedio matemático de cada grupo para encontrar su "centroide" (el punto central representativo de esa clase). 

Después, en la fase de prueba, toma cada dato nuevo, usa la función matemática de la distancia euclidiana para medir qué tan lejos está de cada centroide, y simplemente lo clasifica asignándole la clase del que le quede a la distancia más corta.

In [9]:
def distancia_euclidiana(p1, p2):
    suma = 0.0
    for i in range(len(p1)):
        suma += (p1[i] - p2[i]) ** 2
    return suma ** 0.5

def modelo_distancia_minima(X_train, y_train, X_test):
    clases = np.unique(y_train)
    centroides = []
    
    for c in clases:
        puntos = X_train[y_train == c]
        centroide = np.mean(puntos, axis=0)
        centroides.append((c, centroide))
        
    predicciones = []
    for punto in X_test:
        dist_min = float('inf')
        clase_elegida = -1
        for c, centroide in centroides:
            dist = distancia_euclidiana(punto, centroide)
            if dist < dist_min:
                dist_min = dist
                clase_elegida = c
        predicciones.append(clase_elegida)
        
    return np.array(predicciones)

print("Algoritmo de Distancia Minima")

Algoritmo de Distancia Minima


# Pruebas y Resultados
Ahora sí, ponemos a trabajar al algoritmo de Distancia Mínima con nuestros datasets usando las validaciones de Hold Out (70/30) y K-Fold (k=5).

In [12]:
X_train_b, X_test_b, y_train_b, y_test_b = particion_hold_out(X_bench, y_bench)
X_train_s, X_test_s, y_train_s, y_test_s = particion_hold_out(X_sintetico, y_sintetico)

pred_dm_b = modelo_distancia_minima(X_train_b, y_train_b, X_test_b)
acc_dm_b = accuracy_score(y_test_b, pred_dm_b)

pred_dm_s = modelo_distancia_minima(X_train_s, y_train_s, X_test_s)
acc_dm_s = accuracy_score(y_test_s, pred_dm_s)

kfold_dm_bench = evaluacion_kfold(X_bench, y_bench, 5, modelo_distancia_minima)
kfold_dm_sint = evaluacion_kfold(X_sintetico, y_sintetico, 5, modelo_distancia_minima)

print("--- Distancia Minima ---")
print("Benchmark (Hold Out 70/30):", acc_dm_b)
print("Benchmark (K-Fold k=5):", kfold_dm_bench)
print("Sintetico (Hold Out 70/30):", acc_dm_s)
print("Sintetico (K-Fold k=5):", kfold_dm_sint)

--- Distancia Minima ---
Benchmark (Hold Out 70/30): 0.9555555555555556
Benchmark (K-Fold k=5): 0.9133333333333333
Sintetico (Hold Out 70/30): 0.5222222222222223
Sintetico (K-Fold k=5): 0.5766666666666667


# Modelo 1-NN (Vecino Más Cercano)
A diferencia de Distancia Mínima, el 1-NN no saca promedios. Lo que hace el algoritmo es tomar el punto de prueba y medir la distancia contra absolutamente todos los puntos que tenemos en el entrenamiento. Al final, se queda con la clase del punto que esté más pegadito a él.

In [13]:
def modelo_1nn(X_train, y_train, X_test):
    predicciones = []
    for punto_test in X_test:
        dist_min = float('inf')
        clase_elegida = -1
        for i in range(len(X_train)):
            dist = distancia_euclidiana(punto_test, X_train[i])
            if dist < dist_min:
                dist_min = dist
                clase_elegida = y_train[i]
        predicciones.append(clase_elegida)
    return np.array(predicciones)

print("Algoritmo 1-NN cargado en memoria.")

Algoritmo 1-NN cargado en memoria.


#  1-NN
Validamos el rendimiento de 1-NN con los datasets benchmark y sintético usando Hold Out y K-Fold.

In [14]:
pred_1nn_b = modelo_1nn(X_train_b, y_train_b, X_test_b)
acc_1nn_b = accuracy_score(y_test_b, pred_1nn_b)

pred_1nn_s = modelo_1nn(X_train_s, y_train_s, X_test_s)
acc_1nn_s = accuracy_score(y_test_s, pred_1nn_s)

kfold_1nn_bench = evaluacion_kfold(X_bench, y_bench, 5, modelo_1nn)
kfold_1nn_sint = evaluacion_kfold(X_sintetico, y_sintetico, 5, modelo_1nn)

print("--- Ejecucion terminada: Resultados 1-NN ---")
print("Benchmark (Hold Out 70/30):", acc_1nn_b)
print("Benchmark (K-Fold k=5):", kfold_1nn_bench)
print("Sintetico (Hold Out 70/30):", acc_1nn_s)
print("Sintetico (K-Fold k=5):", kfold_1nn_sint)

--- Ejecucion terminada: Resultados 1-NN ---
Benchmark (Hold Out 70/30): 0.9777777777777777
Benchmark (K-Fold k=5): 0.9533333333333335
Sintetico (Hold Out 70/30): 0.6555555555555556
Sintetico (K-Fold k=5): 0.6666666666666666


In [16]:
import pandas as pd

resultados = {
    "Clasificador": ["Distancia Minima", "Distancia Minima", "1-NN", "1-NN"],
    "Dataset": ["Benchmark (Iris)", "Sintetico (Con Ruido)", "Benchmark (Iris)", "Sintetico (Con Ruido)"],
    "Hold-Out (70/30)": [acc_dm_b, acc_dm_s, acc_1nn_b, acc_1nn_s],
    "K-Fold (k=5)": [kfold_dm_bench, kfold_dm_sint, kfold_1nn_bench, kfold_1nn_sint]
}

tabla_resultados = pd.DataFrame(resultados)
tabla_resultados

,Clasificador,Dataset,Hold-Out (70/30),K-Fold (k=5)
0,Distancia Minima,Benchmark (Iris),0.955556,0.913333
1,Distancia Minima,Sintetico (Con Ruido),0.522222,0.576667
2,1-NN,Benchmark (Iris),0.977778,0.953333
3,1-NN,Sintetico (Con Ruido),0.655556,0.666667


# Conclusion
Al revisar la tabla de resultados, ambos funcionan bien ya que estan ordenados.Pero al meter el dataset sintético con ruido, el rendimiento baja. Esto tiene sentido porque las fronteras entre ambas clases ya no son tan claras.

También me di cuenta de por qué es mejor no confiarse solo del Hold-Out. En algunas ejecuciones, el Hold-Out daba números más altos o más bajos por pura suerte de cómo se partieron los datos al azar en ese 70/30. En cambio, el K-Fold (k=5) nos da una métrica mucho más realista porque promedia el desempeño del modelo iterando sobre varias partes del dataset. 

Finalmente, programarlos desde cero deja muy claro que Distancia Mínima es más rápido a la hora de predecir porque solo compara contra los tres centros, mientras que el 1-NN hace cálculos de distancia contra todo el set de entrenamiento cada vez que clasifica un punto nuevo.